In [ ]:
# Imports
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)

from src import *
import scipy.sparse.linalg as spsl
import matplotlib.pyplot as plt
import matplotlib

In [ ]:
# Hamiltonian Parameters
num_qubits = 15
J = 1
h = 1

# MODMD Parameters
max_energy_level = 3
kd_ratio = 2.5
noise_threshold = 1e-2
epsilon = 1e-3
K_values = np.arange(7, 700+1, 7) + 1
num_modmd_observables = 7
delta_t = 0.08
num_trials = 20

# (U)VQPE Parameters
N_T = 700
steps = np.arange(7, 700+1, 7) + 1
dt = 0.08
r_SVD = 1E-2

In [ ]:
# Generate Hamiltonian and get true eigenenergies
sparse_hamiltonian = tfim_hamiltonian(num_qubits,J,h).to_matrix(sparse=True)

v, w = spsl.eigsh(sparse_hamiltonian,k=20,which = 'SA')
true_eigenenergies = np.unique(np.round(v,8))[:max_energy_level+1]

In [ ]:
# Construct reference state
indices = np.argsort(sparse_hamiltonian.diagonal())
reference_state = bitstring_superposition_state(num_qubits,[bin(indices[i])[2:] for i in range(18)])

# Get evolved reference states
max_K = K_values[-1]
max_d = int(max_K/kd_ratio)
time_evolution_operator = -1j*sparse_hamiltonian*delta_t
evolved_reference_states = spsl.expm_multiply(time_evolution_operator,reference_state,start=0,stop=max_d+max_K+1,num = max_d+max_K+2)

In [ ]:
# MODMD Results and collect measurements for (U)VQPE
identity = [SparsePauliOp('I' * num_qubits).to_matrix(sparse=True)]
H_data = []
S_data = []
modmd_results = []

for trial in range(num_trials):

    modmd_observables = [sparse_hamiltonian] + identity + random_one_local_paulis(num_qubits,num_modmd_observables-1)
    
    left_prods = [np.conj(reference_state)@Oi for Oi in modmd_observables]
    trial_data = []
    for left_prod in left_prods:
        Oi_time_series = []
        for j in range(0, max_d + max_K + 1):
            Oi_time_series.append(left_prod @ evolved_reference_states[j])
        trial_data.append(Oi_time_series)

    trial_data = np.array(trial_data)
    gaussian_noise = np.random.normal(0,epsilon,size=trial_data.shape) + 1j * np.random.normal(0,epsilon,size=trial_data.shape)
    trial_data += gaussian_noise

    H_data.append(trial_data[0])
    S_data.append(trial_data[1])
    
    noisy_X_elements = np.array(trial_data)[1:,:].T.flatten()
    modmd_results.append(varying_K_results(num_modmd_observables,noise_threshold,noisy_X_elements,delta_t,K_values,kd_ratio,max_energy_level))

In [ ]:
# Compute MODMD errors
absolute_modmd_errors = np.array([np.abs(np.array(modmd_results[i]) - true_eigenenergies) for i in range(num_trials)])
Err_MODMD = np.average(absolute_modmd_errors,0).T

In [ ]:
# (U)VQPE results and compute errors
H_row = np.average(H_data,0)
S_row = np.average(S_data,0)

H_row[0], S_row[0] = reference_state@sparse_hamiltonian@reference_state, 1
N_T_max = len(H_data[0])

Err_VQPE = np.zeros((4, len(steps)), dtype=complex)
Err_UVQPE = np.zeros((4, len(steps)), dtype=complex)

for energy_level in range(max_energy_level+1):
    Err_VQPE[energy_level] = np.abs(VQPE(S_row[:N_T+1], H_row[:N_T+1], steps, r_SVD, eigid=energy_level) - true_eigenenergies[energy_level])
    Err_UVQPE[energy_level] = np.abs(UVQPE(N_T, S_row[:N_T+1], steps, r_SVD, eigid=energy_level)/dt - true_eigenenergies[energy_level])

In [ ]:
# Plotting
colors = get_color_set('TFIM')
matplotlib.rcParams.update({'font.size': 14})

for energy_level in range(max_energy_level+1):
    plt.semilogy(K_values + K_values/kd_ratio, Err_UVQPE[energy_level]/Err_MODMD[energy_level], '--',mfc='none', color=colors[energy_level], label=r'$E_{%g}$' % energy_level)
    plt.semilogy(K_values + K_values/kd_ratio, Err_VQPE[energy_level]/Err_MODMD[energy_level], 'x',mfc = 'none', color=colors[energy_level])

plt.xlabel(r'Maximal Simulation Time/$\Delta t$')
plt.ylabel(r'Enhancement'), 
plt.legend(loc='best', edgecolor='none', facecolor='none', fontsize=14)